In [1]:
import argparse
import os
import numpy as np
import pandas as pd

import sys
sys.path.append("..")

from choose_protein_coding import list_of_protein_coding_genes

In [13]:
parser = argparse.ArgumentParser(description="Preprocess GTEx TPM data: align to TCGA gene space and keep protein-coding genes.")
parser.add_argument("--gtex_dir", default="../../data/GTEx", help="Directory containing the raw GTEx .gct file.")
parser.add_argument("--filenamee", default="all_gtex", help="GTEx filename without extension (expects a .gct file).")
parser.add_argument("--tcga_filename", default="../../data/raw_tsv_data/TCGA-GBM.star_tpm.tsv", help="Path to the TCGA TPM file used to define the target gene set.")
parser.add_argument("--gene_info_file", default="../../data/gene_info.csv", help="CSV mapping Ensembl IDs to gene symbols (columns: feature_id, feature_name).")
parser.add_argument("--gene_info_table", default="../../data/gene_info_table.csv", help="CSV with gene metadata including gene_type and ensembl_id columns.")
parser.add_argument("--save_dir", default="../../data/GTEx/processed", help="Output directory for the processed CSV.")

# preprocessing flags
parser.add_argument("--nn", action="store_true", help="No normalization: skip log2(TPM + 1)")
parser.add_argument("--div15", action="store_true", help="Divide data by 1.5")

args = parser.parse_args()

In [15]:
args.filename = "all_gtex"

In [16]:
gtex_filename_path = os.path.join(args.gtex_dir, f"{args.filename}.gct")
df_gtex = pd.read_csv(gtex_filename_path, sep='\t', skiprows=2)
# delete ensembleid versioning
df_gtex['Name'] = df_gtex['Name'].str.split('.').str[0]

# keep only TCGA genes - load TCGA example file
df_tcga = pd.read_csv(args.tcga_filename, sep='\t')
# delete ensemblid versioning
df_tcga['Ensembl_ID'] = df_tcga['Ensembl_ID'].str.split('.').str[0]

# make protein coding lists from knowledge file
df = pd.read_csv(args.gene_info_table, index_col=0)

duplicate_check = df.groupby('ensembl_id')['gene_type'].nunique()
ids_with_multiple_types = duplicate_check[duplicate_check > 1].index.tolist()

if ids_with_multiple_types:
    print(f"Znaleziono duplikaty z różnymi typami dla ID: {ids_with_multiple_types}")
    for gene_id in ids_with_multiple_types:
        names = df[df['ensembl_id'] == gene_id]['gene_name'].unique()
        print(f"  - Gen {gene_id} ({names}) występuje z wieloma typami. Zostanie zachowany jako protein_coding.")

protein_coding_df = df[df['gene_type'] == 'protein_coding'].copy()
protein_coding_df = protein_coding_df.drop_duplicates(subset=['ensembl_id'])

ensembl_list = protein_coding_df['ensembl_id'].tolist()
gene_name_list = protein_coding_df['gene_name'].tolist()

# Printing results
print(f"Number of protein coding ensembl: {len(ensembl_list)}")
print(f"Number of protein coding gene_name: {len(gene_name_list)}")
print("First 5 ID:", ensembl_list[:5])
print("First 5 nazw:", gene_name_list[:5])

valid_ids = set(df_tcga['Ensembl_ID'].unique())
df_gtex_filtered = df_gtex[df_gtex['Name'].isin(valid_ids)].copy()

# Printing results:
print(f"Gene number before filtering: {len(df_gtex)}")
print(f"Gene number after filtering: {len(df_gtex_filtered)}")

lost_genes = len(df_gtex) - len(df_gtex_filtered)
if lost_genes > 0:
    print(f"Attention: {lost_genes} gened from GTEx were lost.")

    gtex_ids = set(df_gtex['Name'].unique())
    missing_ids = gtex_ids - valid_ids
    missing_protein_coding = [gene_id for gene_id in missing_ids if gene_id in ensembl_list]
    print(f"Out of which {len(missing_protein_coding)} were protein coding genes")

# filtering of gtex df
df_gtex = df_gtex[df_gtex['Name'].isin(valid_ids)].reset_index(drop=True)

# choose protein coding in the same way as in TCGA data
df = df_gtex.drop(columns=['Description'])
df = df.T
df.columns = df.iloc[0]
df = df.iloc[1:]

features = pd.read_csv(args.gene_info_file)
id_to_symbol = dict(zip(features["feature_id"], features["feature_name"]))
df = df.rename(columns=id_to_symbol)

df = df.loc[:, df.columns.notnull()]
df = df.loc[:, ~df.columns.duplicated()]

gene_list = df.columns
protein_coding = list_of_protein_coding_genes(gene_list, args.gene_info_table, args.gene_info_file)

Number of protein coding ensembl: 23394
Number of protein coding gene_name: 23394
First 5 ID: ['ENSG00000000003', 'ENSG00000000005', 'ENSG00000000419', 'ENSG00000000457', 'ENSG00000000460']
First 5 nazw: ['TSPAN6', 'TNMD', 'DPM1', 'SCYL3', 'C1orf112']
Gene number before filtering: 41559
Gene number after filtering: 30238
Attention: 11321 gened from GTEx were lost.
Out of which 0 were protein coding genes


In [17]:
df.head()

Name,DDX11L1,MIR1302-2HG,FAM138A,OR4G4P,OR4F5,RP11-34P13.13,CICP27,RP11-34P13.15,RP11-34P13.16,RP11-34P13.14,...,API5,RP11-484D2.2,Y_RNA_ENSG00000252652,TTC17,RP11-484D2.5,RP11-484D2.4,RN7SKP287,PPIAP41,CTBP2P6,MIR670HG
GTEX-1117F-0005-SM-HL9SH,0.0,0.0,0.0,0.0,0.0,0.080554,0.063779,8.0373,73.5519,0.357298,...,2.94645,0.0,0.0,3.38814,0.0,0.315321,0.0,0.0,0.0,0.022785
GTEX-1117F-0011-R10b-SM-GI4VE,0.0,0.0,0.023562,0.0,0.047061,0.114663,0.05674,0.440019,1.87989,0.254294,...,36.646,0.116013,0.0,15.7625,0.0,0.448835,0.308785,0.085559,0.0,1.41084
GTEX-1117F-0011-R11b-SM-GIN8R,0.0,0.027433,0.0,0.073787,0.079804,0.027777,0.024742,0.0,0.94585,0.0,...,37.2482,0.12647,0.0,28.5894,0.0,0.0,0.561031,0.124362,0.0,3.77726
GTEX-1117F-0011-R2b-SM-GI4VL,0.0,0.0,0.012465,0.0,0.066393,0.01348,0.0,0.310386,0.382517,0.0,...,15.4385,0.030688,0.0,5.92456,0.0,0.0,0.0,0.0,0.0,0.360329
GTEX-1117F-0011-R3a-SM-GJ3PJ,0.0,0.027257,0.034029,0.0,0.033983,0.0414,0.016389,0.317745,0.104423,0.183629,...,23.1192,0.0,0.0,10.4921,0.0,0.486167,0.0,0.0,0.0,0.919253


In [18]:
df_small = df.head()
df_small.to_csv('all_gtex_raw_sample.csv')